In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import datetime
import csv

# Configuration
RESULTS_DIR = Path("../results")


In [ ]:
# Path(Path(RESULTS_DIR.parent) / ".." / "measurement")
Path(Path(RESULTS_DIR.parent)).absolute()

In [ ]:
# Define time range for concatenating measurements
start_str = "2026-02-05 10:14:00"  # inclusive start
end_str = "2026-02-05 11:14:04"  

type_meas = "full_swing" # full_swing

fourier_analysis = False
plot_data = True

modulation_frequency = 100e3
acquisition_frequency = 1e6
data_length = 1e4
data_time_span = 1e-3

time_for_quarter_period = 1/modulation_frequency/4
data_points_for_quarter_period = int(time_for_quarter_period*acquisition_frequency)


In [ ]:
%matplotlib widget

In [ ]:
start_dt = datetime.datetime.strptime(start_str, "%Y-%m-%d %H:%M:%S") - datetime.timedelta(seconds=0)
end_dt = datetime.datetime.strptime(end_str, "%Y-%m-%d %H:%M:%S") - datetime.timedelta(seconds=0)
if end_dt <= start_dt:
    raise ValueError("End datetime must be after start datetime")

# Find all files in the specified time range



In [ ]:
def files_list(type_meas, exclude_set = None, start_dt = start_dt, end_dt = end_dt):
    files_in_range = []
    
    # Handle exclude_set: can be None, a list, or a filename (string/Path)
    if exclude_set is None:
        exclude_set = []
    elif isinstance(exclude_set, (str, Path)):
        # If it's a filename, read it and create a list
        exclude_file_path = Path(exclude_set)
        
        # Try to find the file: first as-is, then relative to current directory, then relative to measurement directory
        if not exclude_file_path.is_absolute():
            if not exclude_file_path.exists():
                # Try relative to current directory
                exclude_file_path = Path.cwd() / exclude_file_path
            if not exclude_file_path.exists():
                # Try relative to measurement directory (where RESULTS_DIR is)
                exclude_file_path = RESULTS_DIR.parent / ".." / ".." / exclude_file_path
                print("use relative path")
        
        try:
            with open(exclude_file_path, 'r', encoding='utf-8') as f:
                exclude_set = [line.strip() for line in f if line.strip()]  # Read lines and strip whitespace
            print(f"Loaded {len(exclude_set)} files from exclude list: {exclude_file_path}")
        except FileNotFoundError:
            print(f"Warning: Exclude file not found: {exclude_file_path}. Using empty exclude list.")
            exclude_set = []
        except Exception as e:
            print(f"Warning: Error reading exclude file {exclude_file_path}: {e}. Using empty exclude list.")
            exclude_set = []
    
    current_date = start_dt.date()
    while current_date <= end_dt.date():
        print(current_date)
        for file in sorted((RESULTS_DIR  / current_date.strftime("%Y") / current_date.strftime("%m") / current_date.strftime("%d")).glob(f"{type_meas}_*.json")):
            # Skip if file is in exclusion list
            if file.name in exclude_set:
                print(f"  Excluding: {file.name}")
                continue
            
            # print(file.name)
            try:
                stamp = file.stem.split("_")[-1]  # expects v_meas_YYYYMMDD-HHMMSS.json
                dt = datetime.datetime.strptime(stamp, "%Y%m%d-%H%M%S")
                # print(dt)
            except ValueError:
                continue  # skip files that don't match the timestamp pattern
            if start_dt <= dt <= end_dt:
                print(dt)
                files_in_range.append((dt, file))
        current_date += datetime.timedelta(days=1)
    
    return files_in_range

In [ ]:
# exclude_set = ["full_swing_20260106-000022.json","full_swing_20260106-000310.json","full_swing_20260106-005157.json", 'full_swing_20260106-014647.json', 'full_swing_20260106-021808.json', 'full_swing_20260106-021841.json', 'full_swing_20260106-025901.json',"full_swing_20260106-030327.json"]
exclude_set = []
files_in_range_full_swing = files_list("full_swing", exclude_set = exclude_set)


if not files_in_range_full_swing:
    print("No files found in the specified range.")
else:
    # Load and concatenate data from all files
    concat_time_full_swing = []
    concat_voltage_sin = []
    concat_voltage_cos = []
    concat_voltage_avg = []
    concat_cosine_normalised = []
    concat_voltage_normalised = []
    concat_phase = []
    concat_power = []
    concat_polarisation_measure = []
    concat_error = []
    base_dt = files_in_range_full_swing[0][0]  # anchor absolute time to the first capture

    for dt, file in files_in_range_full_swing:
        with open(file, "r") as f:
            d = json.load(f)
        sampling_rate = d['sample_rate_hz']
        modulation_frequency = d['modulation_frequency_hz']
        time_for_quarter_period = 1/modulation_frequency/4
        data_points_for_quarter_period = int(time_for_quarter_period*sampling_rate)+1
        t = np.array(d["time_s"], dtype=float)
        v = np.array(d["voltage_data_v"], dtype=float)
        v_normalised = (v-np.mean(v))/(np.max(v)-np.min(v))
        cosine_filter = np.array(d["cosine_reference_v"], dtype=float)
        cosine_normalised = (cosine_filter-np.mean(cosine_filter))/(np.max(cosine_filter)-np.min(cosine_filter))
        sine_normalised = list(cosine_normalised[data_points_for_quarter_period+1:]) + list(cosine_normalised[:data_points_for_quarter_period+1])
        sine_normalised = np.array(sine_normalised)
        cosine_phase = sum(v*cosine_normalised)
        sine_phase = sum(v*sine_normalised)
        phase = np.arctan2(sine_phase, cosine_phase)*180/np.pi
        power = np.mean(v)
        polarisation_measure = (max(v) - power)/power
        error = (np.abs(cosine_normalised[-1] - cosine_normalised[0]) + np.abs(sine_normalised[-1] - sine_normalised[0]))/10
        if len(t) == 0 or len(v) == 0:
            continue

        # Offset this capture so its start reflects the true wall-clock interval
        offset = (dt - base_dt).total_seconds()
        concat_time_full_swing.append([(t + offset)[0]])
        # concat_voltage_sin.append([(v*sine_normalised).sum()/len(v)])
        # concat_voltage_cos.append([(v*cosine_filter).sum()/len(v)])
        # concat_voltage_avg.append([(v).sum()])
        # concat_cosine_normalised.append((cosine_normalised).tolist())
        # concat_voltage_normalised.append((v_normalised).tolist())
        concat_phase.append([phase])
        concat_power.append([power])
        concat_polarisation_measure.append([polarisation_measure])
        concat_error.append([error])
    if not concat_time_full_swing:
        print("All files in range were empty after parsing.")
    else:
        concat_time_full_swing = np.concatenate(concat_time_full_swing)
        # concat_voltage_sin = np.concatenate(concat_voltage_sin)
        # concat_voltage_cos = np.concatenate(concat_voltage_cos)
        # concat_voltage_avg = np.concatenate(concat_voltage_avg)
        # concat_cosine_normalised = np.concatenate(concat_cosine_normalised)
        # concat_voltage_normalised = np.concatenate(concat_voltage_normalised)
        concat_phase = np.concatenate(concat_phase)
        concat_power = np.concatenate(concat_power)
        concat_polarisation_measure = np.concatenate(concat_polarisation_measure)
        concat_error = np.concatenate(concat_error)
        print(f"Loaded {len(files_in_range_full_swing)} files")
        print(f"Total data points: {len(concat_time_full_swing)}")
        print(f"Time range: {base_dt} to {base_dt + datetime.timedelta(seconds=concat_time_full_swing[-1])}")



In [ ]:
concat_phase

In [ ]:
concat_power_normalised = (concat_power)/max(concat_power)
concat_polarisation_normalised  = (concat_polarisation_measure)/max(concat_polarisation_measure)
concat_polarization_phase = np.arccos(concat_polarisation_normalised)*180/np.pi
concat_phase_adjusted_winding = np.array([phase + 360 if phase < 0 else phase for phase in concat_phase])


In [ ]:
# Plot concat_power_normalised, concat_polarization_phase, and concat_phase (with error bars)
from datetime import timedelta
import matplotlib.dates as mdates
from matplotlib.dates import SecondLocator, MinuteLocator, HourLocator

if 'concat_time_full_swing' in locals() and len(concat_time_full_swing) > 0:
    concat_datetime = [base_dt + timedelta(seconds=float(t)) for t in concat_time_full_swing]

    fig, axs = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

    # Subplot 1: concat_power_normalised
    axs[0].plot(concat_datetime, concat_power_normalised, linewidth=1, color='tab:blue')
    axs[0].set_ylabel("Normalised Power", fontsize=12)
    axs[0].set_title(f"Data: {start_str} to {end_str} ({len(files_in_range_full_swing)} files)", fontsize=14)
    axs[0].grid(True, alpha=0.3)

    # Subplot 2: concat_polarization_phase
    axs[1].plot(concat_datetime, concat_polarization_phase, linewidth=1, color='tab:orange')
    axs[1].set_ylabel("Polarization Phase (deg)", fontsize=12)
    axs[1].grid(True, alpha=0.3)

    # Subplot 3: concat_phase with error bars
    axs[2].errorbar(concat_datetime, np.array(concat_phase_adjusted_winding),
                    yerr=np.array(concat_error),
                    fmt='-', linewidth=1, elinewidth=0.5, capsize=2,
                    color='tab:green', ecolor='tab:green', alpha=0.8, label='phase +/- error')
    axs[2].set_ylabel("Phase (deg)", fontsize=12)
    axs[2].set_xlabel("Time", fontsize=12)
    axs[2].legend(loc='upper right')
    axs[2].grid(True, alpha=0.3)

    # Format x-axis
    axs[2].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    axs[2].xaxis.set_major_locator(MinuteLocator(interval=10))
    plt.setp(axs[2].xaxis.get_majorticklabels(), rotation=45)

    plt.tight_layout()
    plt.savefig(f"combined_plot_{start_str}_{end_str}.png")
    plt.show()
else:
    print("No data loaded. Please run the data loading cell first.")


In [ ]:
# Plot concat_polarisation_measure
from datetime import timedelta
import matplotlib.dates as mdates
from matplotlib.dates import SecondLocator

if 'concat_time_full_swing' in locals() and len(concat_time_full_swing) > 0:
    concat_datetime = [base_dt + timedelta(seconds=float(t)) for t in concat_time_full_swing]

    fig, ax = plt.subplots(figsize=(12, 4))

    ax.plot(concat_datetime, concat_polarisation_measure, linewidth=1, color='tab:purple', label=r'$\frac{P_{max} - P_{avg}}{P_{avg}}$')
    ax.set_ylabel(r'$\frac{P_{max} - P_{avg}}{P_{avg}}$', fontsize=12)
    ax.set_xlabel("Time", fontsize=12)
    ax.set_title(f"Polarisation Measure: {start_str} to {end_str}", fontsize=14)
    ax.legend(loc='upper right', fontsize=12)
    ax.grid(True, alpha=0.3)

    # Format x-axis
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax.xaxis.set_major_locator(SecondLocator(interval=60))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print("No data loaded. Please run the data loading cell first.")


In [ ]:
# from datetime import timedelta
# k = 2501
# i = 6
# concat_datetime = [base_dt + timedelta(seconds=float(t)) for t in concat_time_full_swing]
# plt.plot(concat_datetime[int(k*i):int(k*i +k)], concat_cosine_normalised[int(k*i):int(k*i +k)], label='interferometry', linewidth=1)
# plt.plot(concat_datetime[int(k*i):int(k*i +k)], concat_voltage_normalised[int(k*i):int(k*i +k)], label='interferometry', linewidth=1)
# plt.show()